# 🐖 Notebook 22 · Entrenamiento de YOLO26n-seg y reproducibilidad experimental

**RECURSO: GPU T4 o superior — entrenamiento.** Notebook independiente: **no ejecutar con CPU**.

**Objetivo:** entrenar YOLO26n-seg con las particiones **congeladas por fecha** de FaroPigSeg utilizadas en el artículo, preservar pesos e historial por semilla, comprobar integridad de datos, y generar documentación de entrenamiento sin tocar las redes anteriores.

### Decisión científica
- El primer paso ejecuta **una semilla piloto, 2026**. El código permite añadir **2027 y 2028** posteriormente, manteniendo carpetas independientes.
- El uso de tres semillas en YOLO26 **no sustituye** las tres semillas de YOLO11 y Mask R-CNN: esa segunda fase se hará por separado.
- Solo se usan **train y val** para aprender y elegir checkpoint. **Test no se evalúa ni se inspecciona aquí.**
- Comparamos un presupuesto nominal parecido a YOLO11 (hasta 60 épocas, imagen 640, lote 8, paciencia 15), **no equivalencia de optimización** entre generaciones ni frente a Mask R-CNN.
- Los análisis realizados con test en notebooks anteriores hacen que futuras comparaciones sobre ese mismo test sean **exploratorias/post hoc**, no una nueva prueba ciega.

Al terminar, ejecuta el notebook de evaluación COCO armonizada que preparemos como **Notebook 23**; no mezcles métricas nativas con el protocolo 09.

## 1 · GPU y dependencias

Asegúrate de activar **Entorno de ejecución → Cambiar tipo de entorno → GPU T4** antes de comenzar.
La versión de Ultralytics queda **fijada** para esta nueva fase. Si la instalación falla, no cambies versiones sin dejarlo documentado.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Se necesita GPU. Colab → Cambiar tipo de entorno → GPU T4.')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA Torch:', torch.version.cuda)

In [ ]:
%pip -q install "ultralytics==8.4.182" "pyyaml>=6,<7" "pandas>=2,<3" "openpyxl>=3.1" "matplotlib>=3.7" pillow

## 2 · Montar Drive y congelar el protocolo de datos

El notebook exige la partición **949 imágenes train, 196 val, 373 test** y **16, 4, 5 fechas**, respectivamente. Las fechas no pueden solaparse entre particiones.
La lista de test se revisa solo para **comprobar integridad del split**, sin leer imágenes ni etiquetas ni realizar inferencia.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, sys, platform, random, zipfile, shutil
import numpy as np, pandas as pd, yaml, matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError('No se encontró la carpeta exacta del proyecto. Comprueba Google Drive.')
SPLIT=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'
YAML=SPLIT/'data_fecha.yaml'
MANIFEST=SPLIT/'particion_fecha_manifest.json'
if not (YAML.is_file() and MANIFEST.is_file()):
    raise FileNotFoundError('Primero deben existir el YAML y el manifiesto congelado del Notebook 02.')
RES=ROOT/'03_Resultados'
TRAIN_DIR=RES/'06_Entrenamiento'/'YOLO26n_fecha_v1'
METRICS_DIR=RES/'03_Metricas'
FIG_DIR=RES/'04_Figuras'
REPORT_DIR=RES/'05_Reportes'/'CERDOS_22_YOLO26_Entrenamiento'
EXPORT_DIR=ROOT/'05_Exportaciones_ZIP'
for p in (TRAIN_DIR,METRICS_DIR,FIG_DIR,REPORT_DIR,EXPORT_DIR):p.mkdir(parents=True,exist_ok=True)

def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

cf=yaml.safe_load(YAML.read_text(encoding='utf-8'))
parts={}
for split in ['train','val','test']:
    pp=Path(cf[split]);assert pp.is_file(),f'Falta lista {split}: {pp}'
    parts[split]=[Path(t.strip()) for t in pp.read_text(encoding='utf-8').splitlines() if t.strip()]
    assert len(parts[split])=={'train':949,'val':196,'test':373}[split],(split,len(parts[split]))
    assert len(set(map(str,parts[split])))==len(parts[split]),'Hay imágenes repetidas en una lista.'
    fechas={x.name[:10] for x in parts[split]}
    assert len(fechas)=={'train':16,'val':4,'test':5}[split],(split,len(fechas))
    print(split,':',len(parts[split]),'imágenes /',len(fechas),'fechas')
assert len(set(map(str,parts['train']))&set(map(str,parts['val'])))==0
assert len(set(map(str,parts['train']))&set(map(str,parts['test'])))==0
assert len(set(map(str,parts['val']))&set(map(str,parts['test'])))==0
assert not ({p.name[:10] for p in parts['train']} & {p.name[:10] for p in parts['val']})
assert not ({p.name[:10] for p in parts['train']} & {p.name[:10] for p in parts['test']})
assert not ({p.name[:10] for p in parts['val']} & {p.name[:10] for p in parts['test']})
# Chequear solo existencia de archivos de entrenamiento/validación y sus polígonos YOLO.
for split in ('train','val'):
    for p in parts[split]:
        if not p.is_file():raise FileNotFoundError('Falta imagen: '+str(p))
        lbl=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        if not lbl.is_file():raise FileNotFoundError('Falta etiqueta: '+str(lbl))
assert cf.get('names') in ({0:'cerdo'},{0:'pig'},['cerdo'],['pig']),cf.get('names')
FREEZE={'yaml_sha256':sha256_file(YAML),'manifest_sha256':sha256_file(MANIFEST),
        'train_txt_sha256':sha256_file(Path(cf['train'])),
        'val_txt_sha256':sha256_file(Path(cf['val'])),
        'test_txt_sha256':sha256_file(Path(cf['test']))}
print('✅ Partición comprobada. Hash YAML:',FREEZE['yaml_sha256'])
print('Ultralytics:',ultralytics.__version__,'| Torch:',torch.__version__)

## 3 · Configuración de los experimentos

Por seguridad se entrena **una semilla por defecto**. Una vez concluida y auditada, se puede activar el bloque de tres semillas; las corridas completadas no vuelven a entrenarse.

**No cambiar a escondidas épocas, tamaño, lote o pesos preentrenados:** toda modificación constituye un experimento distinto y requiere nuevo nombre y protocolo.

In [ ]:
# CONFIGURACIÓN QUE CONTROLA EL INVESTIGADOR
CORRER_TRES_SEMILLAS=True  # Mantener False en el primer experimento.
SEMILLAS=[2026,2027,2028] if CORRER_TRES_SEMILLAS else [2026]
MODELO_BASE='yolo26n-seg.pt'
EPOCAS_MAX=60
BATCH=8
IMGSZ=640
PACIENCIA=15
NUM_WORKERS=2

assert len(set(SEMILLAS))==len(SEMILLAS)
assert ultralytics.__version__=='8.4.182',('Versión inesperada:',ultralytics.__version__)
PROTOCOLO={
    'pipeline':'CERDOS_22_YOLO26_train_seeds_v1',
    'dataset':'FaroPigSeg',
    'split':'split_fecha_v1 / fechas disjuntas',
    'modelo':MODELO_BASE,
    'semillas_configuradas':SEMILLAS,
    'epocas_max':EPOCAS_MAX,'batch':BATCH,'imgsz':IMGSZ,
    'patience':PACIENCIA,'workers':NUM_WORKERS,'deterministic':True,
    'optimizer':'auto (registrado en args.yaml por Ultralytics)',
    'pretrained':'pesos oficiales COCO YOLO26n-seg',
    'device':torch.cuda.get_device_name(0),
    'torch':torch.__version__,'ultralytics':ultralytics.__version__,
    'python':platform.python_version(),
    'frozen_inputs':FREEZE,
    'aviso':'Aquí se usa train/val. El test no se consulta. Comparación con test ya inspeccionado es exploratoria; no validación externa.'
}
REPORT_DIR.joinpath('CERDOS_22_Protocolo.json').write_text(json.dumps(PROTOCOLO,indent=2,ensure_ascii=False),encoding='utf-8')
print('✅ Configuración:',json.dumps({k:PROTOCOLO[k] for k in ['modelo','semillas_configuradas','epocas_max','batch','imgsz']},ensure_ascii=False))

## 4 · Entrenar / reanudar, sin sobrescribir resultados

- Se crea **una carpeta única por semilla**.
- Ante una interrupción de Colab se intenta reanudar desde `last.pt`, si existe y las huellas de los datos coinciden.
- Si una corrida tiene marca de finalización y `best.pt`, **se omite**.
- El mejor checkpoint se selecciona exclusivamente con **validación** por el entrenador de Ultralytics.
- No se deben borrar los archivos `last.pt`, `best.pt`, `results.csv` o `args.yaml`.

⚠️ La ejecución puede tardar bastante según la GPU. Mantén suficiente espacio en Drive.

In [ ]:
from datetime import datetime,timezone
import time

def run_seed(seed):
    exp_name=f'seed_{seed}'
    run=TRAIN_DIR/exp_name
    best=run/'weights'/'best.pt'
    last=run/'weights'/'last.pt'
    done=run/'CERDOS_22_ENTRENAMIENTO_FINALIZADO.json'
    config_path=run/'CERDOS_22_Configuracion_Previa.json'
    required={'seed':int(seed),'model':MODELO_BASE,
              'epocas_max':EPOCAS_MAX,'batch':BATCH,'imgsz':IMGSZ,
              'patience':PACIENCIA,'frozen_inputs':FREEZE,
              'ultralytics':ultralytics.__version__}
    if done.is_file():
        if not best.is_file():raise RuntimeError(f'Marca DONE sin best.pt: {run}')
        old=json.loads(done.read_text(encoding='utf-8'))
        if old['config']!=required:raise RuntimeError('Configuración de corrida completa modificada: '+str(run))
        print(f'✅ {exp_name}: ya finalizado, no se repite.')
        return run
    if config_path.is_file():
        old=json.loads(config_path.read_text(encoding='utf-8'))
        if old!=required:raise RuntimeError('Datos o parámetros cambiaron; no reanudar '+str(run))
    else:
        if last.is_file() or best.is_file():
            raise RuntimeError('Existen pesos sin configuración de control; auditar antes de continuar: '+str(run))
        run.mkdir(exist_ok=True,parents=True)
        config_path.write_text(json.dumps(required,indent=2,ensure_ascii=False),encoding='utf-8')
    start=time.time()
    if last.is_file():
        print(f'♻️ Reanudando {exp_name} desde último checkpoint.')
        YOLO(str(last)).train(resume=True)
    else:
        print(f'🚀 Nuevo entrenamiento: {exp_name}')
        model=YOLO(MODELO_BASE)
        assert model.task=='segment',f'El modelo tiene tarea incorrecta: {model.task}'
        model.train(
            data=str(YAML),epochs=EPOCAS_MAX,imgsz=IMGSZ,batch=BATCH,
            seed=seed,deterministic=True,patience=PACIENCIA,
            device=0,workers=NUM_WORKERS,
            project=str(TRAIN_DIR),name=exp_name,
            exist_ok=True,save=True,save_period=10,plots=True,
            optimizer='auto',amp=True,close_mosaic=10
        )
    if not best.is_file():
        raise RuntimeError('Faltan best.pt al concluir; revisa errores de Colab: '+str(run))
    hist=run/'results.csv'
    if not hist.is_file():
        raise RuntimeError('Falta results.csv; revisar corrida: '+str(run))
    table=pd.read_csv(hist);table.columns=[c.strip() for c in table.columns]
    metadata={
        'config':required,
        'finished_utc':datetime.now(timezone.utc).isoformat(),
        'runtime_sec_this_launch':round(time.time()-start,1),
        'actual_history_rows':len(table),
        'best_weights':str(best),
        'best_weights_sha256':sha256_file(best),
        'training_args_sha256':sha256_file(run/'args.yaml') if (run/'args.yaml').is_file() else None,
        'no_test_used':True
    }
    done.write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
    print(f'✅ Finalizado: {exp_name}. Filas del historial: {len(table)}')
    return run

RUNS=[]
for semilla in SEMILLAS:
    RUNS.append(run_seed(semilla))
print('Corridas consideradas:',*[r.name for r in RUNS])

## 5 · Curvas, informe Excel y trazabilidad

Estas métricas corresponden a **entrenamiento y validación**, nunca a test. El eje de épocas indica los ciclos **realmente ejecutados**, no necesariamente los 60 máximos.

La selección de las mejores épocas la efectúa Ultralytics sobre la validación. La evaluación conjunta de los tres modelos será un notebook posterior que aplique exactamente el mismo protocolo COCO.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

series=[]
for run in RUNS:
    path=run/'results.csv'
    if not path.is_file():continue
    df=pd.read_csv(path);df.columns=[x.strip() for x in df.columns]
    df.insert(0,'seed',int(run.name.split('_')[-1]))
    series.append(df)
if not series:raise RuntimeError('No existen resultados de entrenamiento para resumir.')
full=pd.concat(series,ignore_index=True)

# Las claves exactas se obtienen de la salida de Ultralytics; no se inventan nombres.
for col in full.columns:print('Campo disponible:',col)

def choose(candidates):
    return next((c for c in candidates if c in full.columns),None)
epoch=choose(['epoch']);metric=choose(['metrics/mAP50-95(M)','metrics/mAP50-95(B)'])
if metric!='metrics/mAP50-95(M)':
    raise RuntimeError('Falta la columna de mAP de máscaras. Revisar los encabezados del historial.')
summary=[]
for seed,df in full.groupby('seed'):
    valid=df.dropna(subset=[metric])
    if valid.empty:raise RuntimeError('No existe AP de validación: '+str(seed))
    idx=valid[metric].idxmax()
    summary.append({'semilla':int(seed),'epocas_ejecutadas':len(df),
                    'mejor_epoca_registrada':int(df.loc[idx,epoch]) if epoch else None,
                    'mejor_map_mascaras_val':float(df.loc[idx,metric]),
                    'ruta_best':str(TRAIN_DIR/f'seed_{seed}'/'weights'/'best.pt'),
                    'sha256_best':sha256_file(TRAIN_DIR/f'seed_{seed}'/'weights'/'best.pt')})
resumen=pd.DataFrame(summary)
print('RESUMEN SOLO VAL / TRAIN:')
display(resumen)
excel=METRICS_DIR/'CERDOS_22_YOLO26_Entrenamiento.xlsx'
with pd.ExcelWriter(excel,engine='openpyxl') as writer:
    resumen.to_excel(writer,sheet_name='Resumen',index=False)
    for seed,df in full.groupby('seed'):
        df.to_excel(writer,sheet_name=f'Semilla_{seed}',index=False)
    pd.DataFrame([{'clave':k,'valor':str(v)} for k,v in PROTOCOLO.items()]).to_excel(writer,sheet_name='Protocolo',index=False)

fig,axes=plt.subplots(1,2,figsize=(13,4.8),constrained_layout=True)
for seed,df in full.groupby('seed'):
    t=np.asarray(df[epoch]) if epoch else np.arange(1,len(df)+1)
    train_loss=choose(['train/seg_loss'])
    if train_loss:
        axes[0].plot(t,df[train_loss],label=f'Seed {seed}',linewidth=2)
    axes[1].plot(t,df[metric],label=f'Seed {seed}',linewidth=2)
for a in axes:
    a.xaxis.set_major_locator(MaxNLocator(integer=True))
    a.grid(alpha=.2)
    a.set_xlabel('Epoch')
    a.legend(frameon=False)
axes[0].set(ylabel='Training segmentation loss',title='Training convergence')
axes[1].set(ylabel='Mask AP@[0.50:0.95] (validation)',title='Validation segmentation quality')
fig.suptitle('YOLO26n-seg | FaroPigSeg | date-disjoint train/val',fontsize=14)
figure=FIG_DIR/'CERDOS_22_YOLO26_Curvas_Entrenamiento.png'
fig.savefig(figure,dpi=300,bbox_inches='tight')
plt.show();plt.close(fig)
report={'protocolo':PROTOCOLO,'resumen_semillas':summary,'nota':'Este notebook no evalúa test.'}
(REPORT_DIR/'CERDOS_22_Resumen.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print('✅ Excel:',excel,'\n✅ Figura:',figure)

## 6 · Exportación de resultados (sin duplicar pesos)

Los checkpoints `best.pt`, `last.pt` y las curvas originales se mantienen en Drive, dentro de `03_Resultados/06_Entrenamiento/YOLO26n_fecha_v1/seed_XXXX`. El ZIP contiene **resúmenes** para revisión; no incluye los pesos para evitar copias innecesarias.

In [ ]:
from google.colab import files
out=EXPORT_DIR/'CERDOS_22_YOLO26_Entrenamiento_Resultados.zip'
archives=[excel,figure,REPORT_DIR/'CERDOS_22_Resumen.json',REPORT_DIR/'CERDOS_22_Protocolo.json']
archives += [RUNS[i]/'results.csv' for i in range(len(RUNS))]
archives += [RUNS[i]/'args.yaml' for i in range(len(RUNS)) if (RUNS[i]/'args.yaml').is_file()]
archives += [RUNS[i]/'CERDOS_22_ENTRENAMIENTO_FINALIZADO.json' for i in range(len(RUNS))]
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in archives:
        z.write(p,arcname=(p.parent.name+'__'+p.name))
with zipfile.ZipFile(out) as z:
    assert z.testzip() is None
print('✅ ZIP verificado:',out,'\nTamaño:',round(out.stat().st_size/1024**2,2),'MiB')
print('✅ Pesos permanecen en Drive; no se transfieren en el ZIP.')
files.download(str(out))

## 7 · Cómo interpretar este notebook

Cuando termine, proporciona **`CERDOS_22_YOLO26_Entrenamiento_Resultados.zip`** para auditarlo. Comprueba también que cada semilla tenga pesos `best.pt` y `last.pt` en Drive.

### Aún NO podremos afirmar:
- Que YOLO26 supera a YOLO11 o Mask R-CNN: faltan predicciones de **test con evaluación armonizada**.
- Que el estudio ya es multi-semilla: faltan repeticiones comparables en YOLO11 y Mask R-CNN.
- Que generaliza entre granjas: no hemos realizado **validación externa**.
- Que el resultado es confirmatorio: el conjunto test FaroPigSeg fue consultado anteriormente.

**Siguiente fase:** Notebook 23 — predicciones armonizadas de YOLO26 y comparación estricta con las dos arquitecturas originales; posteriormente, replicaciones y auditoría de convergencia.